# Canonical-State Editing — (pos,vel) fiber collapse + obs-driven probe

**Direction:** `research/directions/canonical-state-editing.md` · `[reframe]` · sub-Q2 (identifiability) + sub-Q3 (editability).

**Premise (verified vs sim).** Dynamics are constant-velocity (`pos_{t+1}=pos_t+vel·dt`, dt=1; velocity never changes — `speed_noise_std=speed/dir = 0`). The minimal sufficient statistic of the world is `(positions, velocities)` = 8-dim for 2 objects. From `(pos,vel)` the optimal rollout is fully determined.

**Hypothesis under test.** Editing fails because the GRU hidden state `h` is **non-canonical**: predictively sufficient but (a) it may carry history *beyond* `(pos,vel)` (uncollapsed decode fiber), and/or (b) it is a *nonlinear* embedding of `(pos,vel)` so linear edits leave the manifold. The readable code ≠ the controllable code.

**Sharpening fact.** The edits split *preserves the teleported object's original velocity*; a min-norm position edit ≈ preserves velocity too. So the ghost is **probably NOT a velocity-incompleteness artifact**. Prediction: if editing the *complete* `(pos,vel)` target still ghosts → culprit is non-canonicality / nonlinear embedding, not incompleteness. Either way is a finding.

**Sections.** A — `(pos,vel)` probe + recoverability. B — fiber-collapse metric `h≈g(pos,vel)`. C — joint `(pos,vel)` editing (obs-space). D — observation-driven editing as a structure probe.

Conventions: numbered code cells `# [N]`, numbered figures `Fig K`. Both rich plots AND printed metric tables. PNGs → `/tmp/canonical_state/`.

---
## 1 — Setup: model, data, teacher-forced state bank, velocities, global subspace

In [ ]:
# [1] Imports + config + checkpoint/data load + teacher-forcing.
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition, identity_mse
from pim.editors import (
    probe_decomposition, inject_state, decompose_hidden,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE      = 512
NUM_WORKERS     = 6
N_OBJ           = 2
SUBSPACE_VAR    = 0.90
LOCAL_K         = 512
LOCAL_VAR       = 0.90
LOCAL_BANK_SIZE = 50_000
OUT = "/tmp/canonical_state"
os.makedirs(OUT, exist_ok=True)

model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
H = model.hidden_size

preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)   # (N,39,H)
print(f"Model : {ckpt_info.run_name} (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"Hidden: {H}   states_tf={states_tf.shape}   device={DEVICE}")
DT = float(test.config["dataset"]["sim"]["dt"]); print("dt =", DT)


In [ ]:
# [2] Velocities read DIRECTLY from HDF5 `velocities` field, aligned like positions[:, :-1].
# states_tf[:, t] aligns with positions[:, t] for t in 0..38, so vel uses [:, :-1, :2, :].
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)   # (N,40,2,2)
vel_tf  = v_test[:, :-1, :, :]                       # (N,39,2,2) aligned with states_tf
pos_tf  = test.positions[:, :-1, :N_OBJ, :]          # (N,39,2,2)
vis_tf  = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N,39) both objects visible

# sanity: constant velocity + finite-difference agreement
fd = (test.positions[:5, 1:, :N_OBJ, :] - test.positions[:5, :-1, :N_OBJ, :]) / DT
print("velocity temporal std (should ~0):", float(v_test.std(axis=1).mean()))
print("|stored vel - finite diff| max (pos noise inflates this):", float(np.abs(fd - v_test[:5, :-1]).max()))
print("vel magnitude  mean|v|=", float(np.abs(vel_tf).mean()), " range=", float(vel_tf.min()), float(vel_tf.max()))
print("pos  magnitude  range=", float(pos_tf.min()), float(pos_tf.max()))

# Stacked 8-dim (pos,vel) target arrays, ordered [obj0_x,obj0_y,obj1_x,obj1_y | vx0,vy0,vx1,vy1]
posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ*2)        # (N,39,4)
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ*2)        # (N,39,4)
posvel_tf  = np.concatenate([posflat_tf, velflat_tf], axis=-1) # (N,39,8)
print("posvel_tf:", posvel_tf.shape)


In [ ]:
# [3] Global state-manifold subspace (PCA) on device + visited-state bank for local tangents.
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace,
    mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
    explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))

_bank_all = states_tf.reshape(-1, H)
_sub = np.random.RandomState(0).choice(_bank_all.shape[0],
        size=min(LOCAL_BANK_SIZE, _bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

real_res_global = float(offmanifold_residual(
    torch.from_numpy(_bank_all[:5000]).float().to(DEVICE), subspace_dev).mean())
def _local_resid(h, n_probe=100):
    res = []
    for i in range(min(n_probe, h.shape[0])):
        sub = fit_local_subspace(bank_dev, h[i], k_neighbors=LOCAL_K,
                                 var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
        res.append(float(offmanifold_residual(h[i:i+1], sub).mean()))
    return float(np.mean(res))
real_res_local = _local_resid(torch.from_numpy(_bank_all[_sub[:200]]).float().to(DEVICE))
print(f"global subspace: kept {subspace.n_components}/{subspace.hidden_size} ({subspace.total_explained:.4f} var)")
print(f"REAL-state off-manifold residual:  global={real_res_global:.4f}  local={real_res_local:.4f}")


---
## Section A — `(pos,vel)` probe + recoverability  [sub-Q2]

Train a probe `h → (pos, vel)` (8-dim). Start **linear**; if velocity is not linearly recoverable, switch to **MLP**; if MLP also fails, investigate temporal features (`h_t − h_{t-1}`, 2-frame windows) and the predicted next obs. Report per-component RMSE and R² vs predict-the-mean baseline.

In [ ]:
# [4] Helper: fit a probe (linear or mlp) on h->target, report per-component RMSE / R2 / baseline.
COMP = ["pos x0","pos y0","pos x1","pos y1","vel x0","vel y0","vel x1","vel y1"]

def fit_report_probe(target_tf, comp_names, kind="linear", mlp_hidden=256, n_epochs=60, lr=3e-3,
                     features=None, feat_name="h", label=""):
    """features: (N,T,F) array used as probe input (default = states_tf). target_tf: (N,T,D)."""
    feats = states_tf if features is None else features
    D = target_tf.shape[-1]
    sdef = StateDefinition(name="posvel", state_shape=(D,), extract_fn=lambda b: b["x"])
    Fdim = feats.shape[-1]
    if kind == "linear":
        probe = LinearExtractor(Fdim, sdef, use_lstsq=True)
        tr = probe.fit(feats, target_tf, mask=vis_tf, device=DEVICE)
    else:
        probe = MLPExtractor(Fdim, sdef, mlp_hidden=mlp_hidden, n_epochs=n_epochs, lr=lr)
        tr = probe.fit(feats, target_tf, mask=vis_tf, device=DEVICE)
    probe = probe.to(DEVICE).eval()
    # predict over masked entries
    with torch.no_grad():
        ft = torch.from_numpy(feats.astype(np.float32)).to(DEVICE)
        pred = probe(ft).cpu().numpy().reshape(*feats.shape[:2], D)
    m = vis_tf
    yt = target_tf[m]; yp = pred[m]                  # (M,D)
    rmse = np.sqrt(((yp-yt)**2).mean(0))
    mu = yt.mean(0); base_rmse = np.sqrt(((yt-mu)**2).mean(0))
    ss_res = ((yp-yt)**2).sum(0); ss_tot = ((yt-mu)**2).sum(0)
    r2 = 1 - ss_res/np.maximum(ss_tot, 1e-12)
    print(f"\n=== PROBE [{label or kind}]  features={feat_name}  train_loss={tr:.6f} ===")
    print(f"{'component':10s} {'RMSE':>9s} {'mean-base':>10s} {'R2':>8s}")
    for j,nm in enumerate(comp_names):
        print(f"{nm:10s} {rmse[j]:9.4f} {base_rmse[j]:10.4f} {r2[j]:8.4f}")
    pos_r2 = r2[:4].mean(); vel_r2 = r2[4:].mean() if D>4 else float('nan')
    print(f"  mean R2  pos={pos_r2:.4f}  vel={vel_r2:.4f}")
    return dict(probe=probe, rmse=rmse, base=base_rmse, r2=r2, pred=pred, train=tr,
                pos_r2=pos_r2, vel_r2=vel_r2)

# A.1 — LINEAR probe on 8-dim (pos,vel)
linA = fit_report_probe(posvel_tf, COMP, kind="linear", label="linear (pos,vel)")


In [ ]:
# [5] A.2 — MLP probe on 8-dim (pos,vel) (in case velocity is only nonlinearly readable).
mlpA = fit_report_probe(posvel_tf, COMP, kind="mlp", mlp_hidden=256, n_epochs=80, lr=2e-3,
                        label="mlp (pos,vel)")


In [ ]:
# [6] A.3 — Temporal-feature probes for velocity (investigate if vel needs h_t - h_{t-1} or a 2-frame window).
# Build temporal features aligned so target is velocity at frame t (t>=1).
dh = states_tf[:, 1:, :] - states_tf[:, :-1, :]        # (N,38,H)  h_t - h_{t-1}
win = np.concatenate([states_tf[:, :-1, :], states_tf[:, 1:, :]], axis=-1)  # (N,38,2H) 2-frame window
vel_t   = velflat_tf[:, 1:, :]                          # (N,38,4) velocity at frame t
vis_t   = vis_tf[:, 1:] & vis_tf[:, :-1]                # both frames visible

# temporarily swap the module-level vis mask + states for the temporal fit
_orig_states, _orig_vis = states_tf, vis_tf
def temporal_probe(feats, kind, label, **kw):
    global states_tf, vis_tf
    states_tf, vis_tf = feats, vis_t
    try:
        return fit_report_probe(vel_t, COMP[4:], kind=kind, features=feats, feat_name=label, label=label, **kw)
    finally:
        states_tf, vis_tf = _orig_states, _orig_vis

print("Velocity from temporal features (target = vel at frame t):")
velLin_dh  = temporal_probe(dh,  "linear", "linear  dh=h_t-h_{t-1}")
velMlp_dh  = temporal_probe(dh,  "mlp",    "mlp     dh=h_t-h_{t-1}", mlp_hidden=256, n_epochs=80, lr=2e-3)
velLin_win = temporal_probe(win, "linear", "linear  [h_{t-1},h_t]")
velMlp_win = temporal_probe(win, "mlp",    "mlp     [h_{t-1},h_t]", mlp_hidden=256, n_epochs=80, lr=2e-3)


In [ ]:
# [7] Fig 1 — Section A recoverability summary: per-component R2 (a) and velocity-R2 across feature sets (b).
plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
x = np.arange(8); w = 0.38
ax.bar(x-w/2, linA["r2"], w, label="linear  h_t", color="#0072B2")
ax.bar(x+w/2, mlpA["r2"], w, label="MLP     h_t", color="#D55E00")
ax.set_xticks(x); ax.set_xticklabels(COMP, rotation=40, ha="right", fontsize=8)
ax.axhline(0, color="k", lw=0.8); ax.axhline(1, color="0.6", ls=":", lw=1)
ax.set_ylabel("R2"); ax.set_title("(a) per-component recoverability from h_t"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

ax = axes[1]
sets = {"lin h_t": linA["r2"][4:].mean(), "MLP h_t": mlpA["r2"][4:].mean(),
        "lin dh": velLin_dh["r2"].mean(), "MLP dh": velMlp_dh["r2"].mean(),
        "lin [h-1,h]": velLin_win["r2"].mean(), "MLP [h-1,h]": velMlp_win["r2"].mean()}
cols = ["#0072B2","#D55E00","#0072B2","#D55E00","#0072B2","#D55E00"]
ax.bar(list(sets.keys()), list(sets.values()), color=cols)
ax.axhline(0, color="k", lw=0.8); ax.axhline(1, color="0.6", ls=":", lw=1)
ax.set_ylabel("mean velocity R2"); ax.set_title("(b) velocity recoverability by feature set")
ax.set_xticklabels(list(sets.keys()), rotation=30, ha="right", fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Fig 1 — Section A: (pos,vel) recoverability from the GRU hidden state", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_recoverability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved fig1_recoverability.png")


---
## Section B — Fiber-collapse metric `h ≈ g(pos,vel)`  [sub-Q2]

How much of `h` is **not** a function of `(pos,vel)`? Regress `h ≈ g(pos,vel)` (linear then MLP) and report residual fraction `‖h−g‖/‖h‖` and R² on `h`. Residual≈0 ⇒ canonical (fiber collapsed); large residual ⇒ `h` carries history beyond the sufficient statistic. Compare `g(pos)` vs `g(pos,vel)` for the velocity increment; linear-vs-MLP gap measures embedding nonlinearity.

In [ ]:
# [8] Fit g: (input)->h, linear then MLP. Report residual fraction + R2 on h.  inputs: pos, posvel.
m = vis_tf
H_tgt = states_tf[m]                                   # (M,H)
h_norm2 = (H_tgt**2).sum()

def fit_g(inp_tf, kind, hidden=512, n_epochs=120, lr=1.5e-3):
    X = inp_tf[m]                                       # (M, Din)
    Din = X.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
    Yt = torch.from_numpy(H_tgt.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0],1,device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din,hidden), nn.ReLU(),
                            nn.Linear(hidden,hidden), nn.ReLU(),
                            nn.Linear(hidden,H)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]
                loss = ((net(Xt[idx]) - Yt[idx])**2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt)**2).sum()
    res_frac = float((resid2 / h_norm2).sqrt())          # ||h-g|| / ||h||
    # R2 on h (per-dim, aggregated)
    mu = Yt.mean(0, keepdim=True)
    r2 = float(1 - resid2 / ((Yt-mu)**2).sum())
    return res_frac, r2

print(f"{'g model':22s} {'resid frac ||h-g||/||h||':>24s} {'R2 on h':>10s}")
res = {}
for name, inp in [("pos (4d)", posflat_tf), ("pos,vel (8d)", posvel_tf)]:
    for kind in ["linear","mlp"]:
        rf, r2 = fit_g(inp, kind)
        res[(name,kind)] = (rf, r2)
        print(f"{name+' '+kind:22s} {rf:24.4f} {r2:10.4f}")


In [ ]:
# [9] Section B summary table + interpretation numbers.
print("=== FIBER-COLLAPSE SUMMARY ===")
print(f"{'':16s} {'linear R2(h)':>13s} {'MLP R2(h)':>11s} {'lin resid':>11s} {'MLP resid':>11s}")
for name in ["pos (4d)","pos,vel (8d)"]:
    lrf,lr2 = res[(name,'linear')]; mrf,mr2 = res[(name,'mlp')]
    print(f"{name:16s} {lr2:13.4f} {mr2:11.4f} {lrf:11.4f} {mrf:11.4f}")
inc_lin = res[('pos,vel (8d)','linear')][1] - res[('pos (4d)','linear')][1]
inc_mlp = res[('pos,vel (8d)','mlp')][1]    - res[('pos (4d)','mlp')][1]
gap_pos = res[('pos (4d)','linear')][0]      - res[('pos (4d)','mlp')][0]
gap_pv  = res[('pos,vel (8d)','linear')][0]  - res[('pos,vel (8d)','mlp')][0]
print(f"\nincremental R2(h) from adding velocity:  linear {inc_lin:+.4f}   MLP {inc_mlp:+.4f}")
print(f"linear->MLP residual drop (embedding nonlinearity):  pos {gap_pos:+.4f}   pos,vel {gap_pv:+.4f}")
print(f"MLP residual fraction with full (pos,vel): {res[('pos,vel (8d)','mlp')][0]:.4f}")
print("  -> residual~0 => h is a function of (pos,vel) (CANONICAL); large => h carries extra history")


In [ ]:
# [10] Fig 2 — Fiber-collapse: residual fraction (a) and R2 on h (b), linear vs MLP, pos vs pos+vel.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
labels = ["pos (4d)","pos,vel (8d)"]; xx = np.arange(2); w=0.38
ax = axes[0]
ax.bar(xx-w/2, [res[(l,'linear')][0] for l in labels], w, label="linear g", color="#0072B2")
ax.bar(xx+w/2, [res[(l,'mlp')][0]    for l in labels], w, label="MLP g",    color="#D55E00")
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylabel("residual fraction  ||h-g||/||h||")
ax.set_title("(a) is h determined by (pos,vel)?"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
ax = axes[1]
ax.bar(xx-w/2, [res[(l,'linear')][1] for l in labels], w, label="linear g", color="#0072B2")
ax.bar(xx+w/2, [res[(l,'mlp')][1]    for l in labels], w, label="MLP g",    color="#D55E00")
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylabel("R2 on h"); ax.set_ylim(0,1.02)
ax.set_title("(b) variance of h explained by (pos,vel)"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Fig 2 — Section B: decode-fiber collapse  (h ≈ g(pos,vel))", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_fiber_collapse.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved fig2_fiber_collapse.png")


---
## Section C — Joint `(pos,vel)` editing  [sub-Q3]

Edit the **complete** sufficient statistic. Target = post-edit `(pos, original_vel)` at `edit_frame` (positions teleported per `edits`; velocities = edits' preserved original velocities). Inject via the 8-dim `(pos,vel)` linear probe (min-norm), plus a global-manifold variant. Head-to-head vs **position-only** edit. Measure in obs space (→target render, obs change, ghost ratio) + 1D scans + waterfalls. **Headline: does completing the target to `(pos,vel)` fix the ghost / move the object?**

In [ ]:
# [11] Warm up to edit frame; build position-only and (pos,vel) probes + targets.
N_EDIT    = 64
N_ROLLOUT = 15
N = min(N_EDIT, edits.n_samples)

warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame, n_viz=N, n_ctx_show=8, device=DEVICE)
h_base = warm.h_at_edit[:N]
h0 = torch.from_numpy(h_base).float().to(DEVICE)

# ---- position-only linear probe (the existing baseline), fit on (pos) only ----
pos_sdef = StateDefinition(name="positions", state_shape=(N_OBJ,2), extract_fn=lambda b: b["positions"])
linear_pos = LinearExtractor(H, pos_sdef, use_lstsq=True)
linear_pos.fit(states_tf, pos_tf, mask=vis_tf, device=DEVICE)
linear_pos = linear_pos.to(DEVICE).eval()
Ap, bp, Ap_pinv = probe_decomposition(linear_pos)

# ---- joint (pos,vel) linear probe (reuse linA from Section A) ----
linear_pv = linA["probe"]
Apv, bpv, Apv_pinv = probe_decomposition(linear_pv)

# ---- velocities for the edits split, aligned: edit-frame velocity = preserved original velocity ----
v_edits = h5py.File(edits.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)  # (N,40,2,2)
ef = edits.edit_frame
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)            # teleported positions
tgt_vel = v_edits[:N, ef, :, :].astype(np.float32)                        # preserved velocity
targets_pos = tgt_pos.reshape(N, N_OBJ*2)                                  # (N,4)
targets_pv  = np.concatenate([targets_pos, tgt_vel.reshape(N, N_OBJ*2)], 1)  # (N,8)

print(f"N={N}  edit_frame={ef}  h_base={h_base.shape}")
print(f"linear_pos train fit done; linear_pv reused from Section A")
print(f"target pos sample0:\n{tgt_pos[0]}\ntarget vel sample0:\n{tgt_vel[0]}")


In [ ]:
# [12] Build edited states: position-only, joint (pos,vel), and global-manifold (pos,vel) variant.
POCS_ITERS = 50
tgt_pos_t = torch.from_numpy(targets_pos).float().to(DEVICE)
tgt_pv_t  = torch.from_numpy(targets_pv).float().to(DEVICE)

# (a) position-only pseudo-inverse edit
h_posonly = inject_state(h0, tgt_pos_t, Ap, Ap_pinv, bp)
# (b) joint (pos,vel) pseudo-inverse edit
h_posvel  = inject_state(h0, tgt_pv_t, Apv, Apv_pinv, bpv)
# (c) joint (pos,vel) global-manifold edit (POCS against global PCA)
edit_fn_pv = lambda h, t: inject_state(h, t, Apv, Apv_pinv, bpv)
h_posvel_mani = manifold_steer(h0, tgt_pv_t, edit_fn_pv, subspace_dev, n_iters=POCS_ITERS)

def resid_global(h): return float(offmanifold_residual(h, subspace_dev).mean())
def pos_rmse(h):  return float(((h @ Ap.T + bp) - tgt_pos_t).pow(2).mean().sqrt())
def pv_rmse(h):   return float(((h @ Apv.T + bpv) - tgt_pv_t).pow(2).mean().sqrt())

print("=== EDITED-STATE READOUT / OFF-MANIFOLD TABLE ===")
print(f"{'variant':18s} {'pos RMSE':>9s} {'posvel RMSE':>12s} {'glob resid':>11s} {'loc resid':>10s}")
edit_states = {"unsteered":h0, "pos-only":h_posonly, "pos,vel":h_posvel, "pos,vel manifold":h_posvel_mani}
for nm,h in edit_states.items():
    print(f"{nm:18s} {pos_rmse(h):9.4f} {pv_rmse(h):12.4f} {resid_global(h):11.4f} {_local_resid(h):10.4f}")
print(f"{'real states':18s} {'—':>9s} {'—':>12s} {real_res_global:11.4f} {real_res_local:10.4f}")


In [ ]:
# [13] Roll out each edited state; build TARGET / PRE renders for obs-space metrics.
@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout); obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

variant_h = {"unsteered":h0, "pos-only":h_posonly, "pos,vel":h_posvel, "pos,vel manifold":h_posvel_mani}
roll_obs, roll_hs = {}, {}
for nm,h in variant_h.items():
    o,hs = rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT); roll_obs[nm]=o; roll_hs[nm]=hs
OBS_RES = roll_obs["unsteered"].shape[-1]
print("rollouts:", {k:v.shape for k,v in roll_obs.items()})

from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
sim = test.config["dataset"]["sim"]
def make_cfg(n_frames):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"],
                     n_frames=n_frames, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                     fixed_reflectivities=True, obs_noise_std=0.0, boundary="open",
                     always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1,1,1]],dtype=np.float32),(N_OBJ,1))
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)
cfg1 = make_cfg(1)
tgt_render_id  = np.zeros((N,OBS_RES), np.int64); tgt_render_int = np.zeros((N,OBS_RES), np.float32)
pre_render_id  = np.zeros((N,OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i],tgt_render_int[i]=rid[0],rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i]=ridp[0]
print("renders built:", tgt_render_int.shape)


In [ ]:
# [14] Obs-space metrics: ->target render, obs change vs unsteered, ghost ratio. Table over rollout step 0.
obs_u = roll_obs["unsteered"]; edit_obj = edits.edit_object[:N]
def rms(a,b): return float(np.sqrt(((a-b)**2).mean()))
def dist_to_target(obs, s=0): return rms(obs[:,s,:], tgt_render_int)
def obs_change(obs, s=0):     return rms(obs[:,s,:], obs_u[:,s,:])
unsteered_to_target = dist_to_target(obs_u)
ghost_mask = np.zeros((N,OBS_RES), bool)
for i in range(N):
    ghost_mask[i] = (pre_render_id[i]==edit_obj[i]) & (tgt_render_id[i]!=edit_obj[i])
ghost_denom = ghost_mask.sum()
def ghost_ratio(obs, s=0):
    if ghost_denom==0: return np.nan
    return float(obs[:,s,:][ghost_mask].mean() / max(obs_u[:,s,:][ghost_mask].mean(),1e-6))
print(f"ghost rays available: {int(ghost_denom)}  unsteered intensity there={obs_u[:,0,:][ghost_mask].mean():.3f}")
print(f"reference gap unsteered->target (step0) = {unsteered_to_target:.4f}\n")
print("=== OBS-SPACE TABLE (step 0 = direct edit) ===")
print(f"{'variant':18s} {'->target':>9s} {'obs chg':>9s} {'ghost':>7s} {'%gap closed':>12s}")
obs_table = {}
for nm,obs in roll_obs.items():
    dt_,dc,g = dist_to_target(obs), obs_change(obs), ghost_ratio(obs)
    frac = 100*(unsteered_to_target-dt_)/unsteered_to_target if unsteered_to_target>1e-6 else 0
    obs_table[nm]=(dt_,dc,g)
    print(f"{nm:18s} {dt_:9.4f} {dc:9.4f} {g:7.3f} {frac:12.1f}")


In [ ]:
# [15] Fig 3 — Section C obs-space step curves: (a) ->target render (b) obs change (c) ghost ratio.
steps = np.arange(N_ROLLOUT)
to_tgt_step = {n:[dist_to_target(o,s) for s in steps] for n,o in roll_obs.items()}
chg_step    = {n:[obs_change(o,s)     for s in steps] for n,o in roll_obs.items()}
ghost_step  = {n:[ghost_ratio(o,s)    for s in steps] for n,o in roll_obs.items()}
COL = {"unsteered":"0.5","pos-only":"#D55E00","pos,vel":"#0072B2","pos,vel manifold":"#009E73"}
MK  = {"unsteered":None,"pos-only":"v","pos,vel":"o","pos,vel manifold":"s"}
fig, axes = plt.subplots(1,3, figsize=(16,4.4))
for ax,(d,ylab,ttl) in zip(axes,
        [(to_tgt_step,"RMS(gen obs, TARGET render)","(a) does the obs reach the target?"),
         (chg_step,"RMS obs change vs unsteered","(b) did the edit move the output?"),
         (ghost_step,"ghost ratio (pre-edit loc)","(c) is the ghost gone?")]):
    for n,v in d.items(): ax.plot(steps,v,color=COL[n],marker=MK[n],ms=3,lw=2.0 if n.startswith("pos,vel") else 1.3,label=n)
    ax.set_xlabel("rollout step"); ax.set_ylabel(ylab); ax.set_title(ttl); ax.grid(alpha=0.3); ax.legend(fontsize=7)
axes[2].axhline(1.0,color="0.7",ls="--",lw=1); axes[2].axhline(0.0,color="0.7",ls=":",lw=1)
fig.suptitle("Fig 3 — Section C: joint (pos,vel) vs position-only editing in observation space", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_C_obs_metrics.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig3_C_obs_metrics.png")


In [ ]:
# [16] Fig 4 — Section C: 1D scans (a) + waterfalls (b) at the direct-edit step. Green=target, red=ghost.
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport*has_ghost)[::-1][:3])
print("samples:", SAMPLES, "teleport=", [round(float(teleport[s]),2) for s in SAMPLES])
rays = np.arange(OBS_RES); order = ["unsteered","pos-only","pos,vel","pos,vel manifold"]

# (a) scans
fig, axes = plt.subplots(len(SAMPLES),1, figsize=(11,3.0*len(SAMPLES)), squeeze=False)
for r,smp in enumerate(SAMPLES):
    ax = axes[r][0]
    ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="TARGET render", zorder=5)
    gz = np.where(ghost_mask[smp])[0]
    if gz.size: ax.axvspan(gz.min()-.5, gz.max()+.5, color="red", alpha=0.10, label="ghost zone")
    for n in order:
        ax.plot(rays, roll_obs[n][smp,0], color=COL[n], lw=2.2 if n.startswith("pos,vel") else 1.3,
                alpha=0.95 if n.startswith("pos,vel") else 0.8, label=n)
    ax.set_title(f"sample {smp} (obj {edit_obj[smp]}, teleport={teleport[smp]:.2f})")
    ax.set_xlabel("ray"); ax.set_ylabel("intensity"); ax.set_ylim(-.02,1.05); ax.grid(alpha=.25); ax.legend(fontsize=7,ncol=3)
fig.suptitle("Fig 4a — Section C generated 1D scans at direct-edit step vs TARGET render", y=1.005, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4a_C_scans.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

# (b) waterfalls
def centroid(mask_row):
    idx=np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
fig, axes = plt.subplots(len(SAMPLES), len(order), figsize=(2.6*len(order),3.0*len(SAMPLES)), squeeze=False)
for r,smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_render_id[smp]==edit_obj[smp]); pre_cx = centroid(pre_render_id[smp]==edit_obj[smp])
    for c,n in enumerate(order):
        ax=axes[r][c]; ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4)
        if r==0: ax.set_title(n, fontsize=9)
        if c==0: ax.set_ylabel(f"smp {smp}\nframe", fontsize=9)
        ax.set_xlabel("ray", fontsize=8)
fig.suptitle("Fig 4b — Section C waterfalls: green=where edited obj SHOULD be, red=ghost zone", y=1.01, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4b_C_waterfalls.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)
print("saved fig4a_C_scans.png, fig4b_C_waterfalls.png")


---
## Section D — Observation-driven editing as a structure probe  [sub-Q3]

Drive `h` so the model's **generated** observation matches a target, then study *where it lands*. This is an **oracle** probe of structure (uses GT obs to *find* a latent), NOT a deployable editor. Two targets: a **single GT edit frame** (under-determines velocity) and a **short GT sequence** (pins velocity). Optimizer: Adam on `h` minimizing `‖decode(state_from_flat(h))−target‖²` (single) / differentiable k-step rollout (sequence). At `h*` report: (a) `(pos,vel)` probe readout vs GT; (b) off-manifold residual; (c) distance to the real canonical state; (d) does the rollout stick?

In [ ]:
# [17] Differentiable obs-driven optimizers: single-frame and k-step sequence (Adam on h).
# NOTE: cudnn RNN backward requires train mode; we disable cudnn so backward works on the eval GRU
# (numerically identical forward, just no fused RNN kernel). model params stay frozen — we only optimize h.
TARGET_OBJ_DT = "single & sequence"
def obs_opt_single(h_init, target_obs, n_iter=400, lr=0.05):
    h = h_init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            state = model.state_from_flat(h)
            pred = model.decode(state)
            loss = ((pred - target_obs)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())

def obs_opt_sequence(h_init, target_seq, n_iter=400, lr=0.05):
    # target_seq: (B, K, R). step 0 = decode(h); steps 1..K-1 via predict_step.
    K = target_seq.shape[1]
    h = h_init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            state = model.state_from_flat(h)
            preds = [model.decode(state)]
            for _k in range(K-1):
                p, state = model.predict_step(state); preds.append(p)
            pred = torch.stack(preds, 1)                 # (B,K,R)
            loss = ((pred - target_seq)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())

# GT target obs: use the edits split's actual (clean) observations at/after the edit frame.
K_SEQ = 5
gt_single = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)          # (N,R)
gt_seq    = torch.from_numpy(edits.clean_obs[:N, ef:ef+K_SEQ, :]).float().to(DEVICE) # (N,K,R)

# unsteered baseline reconstruction loss (for context)
with torch.no_grad():
    base_single = float(((model.decode(model.state_from_flat(h0)) - gt_single)**2).mean())
print(f"unsteered decode vs single-frame GT obs: {base_single:.6f}")

h_obs_single, l_s = obs_opt_single(h0, gt_single)
h_obs_seq,    l_q = obs_opt_sequence(h0, gt_seq)
print(f"obs-driven single-frame final loss = {l_s:.6f}")
print(f"obs-driven sequence(K={K_SEQ}) final loss = {l_q:.6f}")


In [ ]:
# [18] Canonical reference state: the REAL teacher-forced h that genuinely produces the post-edit obs.
# Teacher-force the FULL edits obs sequence; the hidden state at edit_frame is the canonical state.
@torch.no_grad()
def tf_hidden_at(obs_seqs, frame):
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state=None
        for t in range(frame+1):
            _, state = model.step(ot[t].unsqueeze(0), state)
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out
# Canonical = hidden state after teacher-forcing the post-edit GT trajectory up TO edit frame.
# (edits.obs already contains the realized post-edit observations.)
h_canon = torch.from_numpy(tf_hidden_at(edits.obs[:N], ef)).float().to(DEVICE)

def dist_to_canon(h): return float((h - h_canon).norm(dim=1).mean())
def readout_pv(h):    return (h @ Apv.T + bpv)
def pv_err_to_gt(h):  return float((readout_pv(h) - tgt_pv_t).pow(2).mean().sqrt())
def pos_err_to_gt(h): return float(((h @ Ap.T + bp) - tgt_pos_t).pow(2).mean().sqrt())

print("=== SECTION D ENDPOINT TABLE ===")
print(f"{'state':20s} {'pos RMSE(gt)':>12s} {'posvel RMSE(gt)':>15s} {'glob resid':>11s} {'loc resid':>10s} {'dist->canon':>12s}")
D_states = {"unsteered":h0, "probe pos,vel (C)":h_posvel, "obs single":h_obs_single,
            "obs seq":h_obs_seq, "canonical (real)":h_canon}
for nm,h in D_states.items():
    print(f"{nm:20s} {pos_err_to_gt(h):12.4f} {pv_err_to_gt(h):15.4f} {resid_global(h):11.4f} {_local_resid(h):10.4f} {dist_to_canon(h):12.4f}")


In [ ]:
# [19] Roll out Section-D endpoints; obs-space stick test (->target render / obs change / ghost).
variant_h_D = {"unsteered":h0, "probe pos,vel":h_posvel, "obs single":h_obs_single,
               "obs seq":h_obs_seq, "canonical":h_canon}
roll_obs_D = {}
for nm,h in variant_h_D.items():
    o,_ = rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT); roll_obs_D[nm]=o
print("=== SECTION D OBS-SPACE TABLE (does it STICK?) ===")
print(f"{'variant':16s} {'->tgt s0':>9s} {'->tgt s4':>9s} {'obs chg s0':>11s} {'ghost s0':>9s} {'ghost s4':>9s}")
for nm,obs in roll_obs_D.items():
    g0 = ghost_ratio(obs,0); g4 = ghost_ratio(obs,4) if N_ROLLOUT>4 else np.nan
    print(f"{nm:16s} {dist_to_target(obs,0):9.4f} {dist_to_target(obs,4):9.4f} {obs_change(obs,0):11.4f} {g0:9.3f} {g4:9.3f}")


In [ ]:
# [20] Fig 5 — Section D: (a) endpoint geometry bars, (b) obs->target stick curves, (c) waterfalls.
fig = plt.figure(figsize=(16, 9))
# (a) geometry: pv error to GT, global resid, dist->canon (normalized columns)
ax = fig.add_subplot(2,2,1)
names = ["unsteered","probe pos,vel (C)","obs single","obs seq","canonical (real)"]
pv_e  = [pv_err_to_gt(D_states[n]) for n in names]
gres  = [resid_global(D_states[n]) for n in names]
dcan  = [dist_to_canon(D_states[n]) for n in names]
xx=np.arange(len(names)); w=0.26
ax.bar(xx-w, pv_e, w, label="posvel RMSE -> GT", color="#0072B2")
ax.bar(xx,   gres, w, label="global off-manifold resid", color="#D55E00")
ax.bar(xx+w, dcan, w, label="dist -> canonical h", color="#009E73")
ax.axhline(real_res_global, color="#D55E00", ls=":", lw=1)
ax.set_xticks(xx); ax.set_xticklabels(names, rotation=25, ha="right", fontsize=8)
ax.set_title("(a) endpoint geometry"); ax.legend(fontsize=7); ax.grid(alpha=.3, axis="y")
# (b) stick curves
ax = fig.add_subplot(2,2,2)
COLD = {"unsteered":"0.5","probe pos,vel":"#0072B2","obs single":"#CC79A7","obs seq":"#E69F00","canonical":"#009E73"}
for n,obs in roll_obs_D.items():
    ax.plot(steps, [dist_to_target(obs,s) for s in steps], color=COLD[n], marker="o", ms=3, label=n)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS(gen obs, TARGET render)")
ax.set_title("(b) does the obs reach target & STICK?"); ax.grid(alpha=.3); ax.legend(fontsize=7)
# (c)+(d) waterfalls for top sample: obs single vs obs seq vs canonical vs probe
smp = SAMPLES[0]
tgt_cx = centroid(tgt_render_id[smp]==edit_obj[smp]); pre_cx = centroid(pre_render_id[smp]==edit_obj[smp])
wf_order = ["probe pos,vel","obs single","obs seq","canonical"]
for j,n in enumerate(wf_order):
    ax = fig.add_subplot(2,4,5+j)
    ax.imshow(roll_obs_D[n][smp], aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4)
    if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4)
    ax.set_title(n, fontsize=9); ax.set_xlabel("ray", fontsize=8)
    if j==0: ax.set_ylabel(f"smp {smp}\nframe", fontsize=9)
fig.suptitle("Fig 5 — Section D: obs-driven endpoints (single vs sequence) vs probe-edit vs canonical", y=1.0, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_D_obs_driven.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig5_D_obs_driven.png")


---
## Summary

See `research/scratch/2026-06-24-canonical-state-editing.md` for the verdict on the hypothesis. Key questions answered per section: A — is velocity recoverable (and how)? B — is the `(pos,vel)` fiber collapsed (canonical)? C — does completing the target to `(pos,vel)` fix the ghost / move the obs? D — does obs-driven editing land on the manifold / on the canonical state, and stick?